In [138]:
pwd

'/Users/lubina.zaidi/Documents/Claims audit and fraud detection'

In [141]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler, MinMaxScaler
from sklearn.ensemble import IsolationForest
from scipy.stats import entropy

In [142]:
df = pd.read_excel('consolidated_Vishalquery_detail_data.xlsx')
#consolidated_Vishalquery_detail_data = df


Claims audit & fraud detection — feature summary
Bike - Claim-level Stat Validation

In [143]:
df.head(4)


,ClaimID,InvoiceDate,ClaimDate,ClaimItemId,ItemClaimedDate,ItemDescription,ItemQuantity,ClaimItemStatusDate,ClaimantName,ClaimantName.1,...,ProductName.1,ProductDescription,ProductManafacturer,SerialNumber,ClaimItemStatus,ClaimItemStatusReason,SalesTransactionID,SaleQuantity,SaleAmount,AuditComment
0,351322,2022-02-17 06:00:00,NaN,351322,2022-02-18 18:12:23.856,NaN,1,2022-02-18 18:27:11.282,Sheila Mundey,Sheila Mundey,...,BIZHUB 654E PRINTER COPIER,NaN,NaN,a5yn017008872,Cancelled,NaN,NaN,NaN,NaN,NaN
1,351472,2021-08-10 05:00:00,NaN,351472,2022-03-02 14:48:57.054,NaN,1,2022-03-02 14:48:57.054,Christian Colasono,Christian Colasono,...,BIZHUB C750I PRINTER COPIER,NaN,NaN,ACKN011001276,Processed,Select For Audit,386753.0,1.0,NaN,NaN
2,351483,2022-02-01 06:00:00,NaN,351483,2022-02-24 14:55:45.287,NaN,1,2022-02-24 14:55:45.287,Ryan Eades,Ryan Eades,...,ACCURIOPRESS C4080,NaN,NaN,AC57011020078,Processed,Select For Audit,351719.0,1.0,NaN,NaN
3,351182,2022-02-09 06:00:00,NaN,351182,2022-02-16 10:24:56.946,NaN,1,2022-02-16 12:28:03.430,Admin User,Admin User,...,ACCURIOPRESS C3080P,NaN,NaN,12314344445,Cancelled,NaN,NaN,NaN,NaN,NaN


In [47]:
df.columns

Index(['ClaimID', 'InvoiceDate', 'ClaimDate', 'ClaimItemId', 'ItemClaimedDate',
       'ItemDescription', 'ItemQuantity', 'ClaimItemStatusDate',
       'ClaimantName', 'ClaimantName.1', 'ClaimName', 'ProductSoldDate',
       'ClaimSubmitionDate', 'DealerName', 'ProductID', 'ProductStatus',
       'ProductName', 'IsSerializedProduct', 'ProductName.1',
       'ProductDescription', 'ProductManafacturer', 'SerialNumber',
       'ClaimItemStatus', 'ClaimItemStatusReason', 'SalesTransactionID',
       'SaleQuantity', 'SaleAmount', 'AuditComment'],
      dtype='str')

In [76]:
# # compare ItemClaimedDate and ClaimItemStatusDate - No of rows where both are NaT, No of rows where both are equal, No of rows where both are different
# # ProductSoldDate and invoiceDate - No of rows where both are NaT, No of rows where both are equal, No of rows where both are different
# # invoiceDate and ClaimDate - No of rows where both are NaT, No of rows where both are equal, No of rows where both are different

# a = pd.to_datetime(df['InvoiceDate'], errors='coerce')
# b = pd.to_datetime(df['ItemClaimedDate'], errors='coerce')

# both_nat = a.isna() & b.isna()
# equal_vals = a == b
# equal_or_both_na = equal_vals | both_nat

# print("total_rows:", len(df))
# print("both_NaT:", both_nat.sum())
# print("exact_equal_including_both_NaT:", equal_or_both_na.sum())
# print("exact_equal_excluding_both_NaT:", equal_vals.sum())
# print("different_count:", len(df) - equal_or_both_na.sum())

# # show sample mismatches
# mismatches = df.loc[~equal_or_both_na, ['ClaimID', 'InvoiceDate', 'ClaimDate']]
# mismatches.head(10)

In [48]:
len(df)

128805

In [49]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 128805 entries, 0 to 128804
Data columns (total 28 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   ClaimID                128805 non-null  int64         
 1   InvoiceDate            127608 non-null  datetime64[us]
 2   ClaimDate              0 non-null       float64       
 3   ClaimItemId            128805 non-null  int64         
 4   ItemClaimedDate        125081 non-null  datetime64[us]
 5   ItemDescription        0 non-null       float64       
 6   ItemQuantity           128805 non-null  int64         
 7   ClaimItemStatusDate    128805 non-null  datetime64[us]
 8   ClaimantName           128805 non-null  str           
 9   ClaimantName.1         128805 non-null  str           
 10  ClaimName              128805 non-null  str           
 11  ProductSoldDate        127608 non-null  datetime64[us]
 12  ClaimSubmitionDate     125081 non-null  datetime64[us]


In [50]:
df.ProductStatus.value_counts()

ProductStatus
A    128634
I       171
Name: count, dtype: int64

In [113]:
df1 = df.ProductName.value_counts().reset_index().rename(columns={'index': 'ProductName', 'ProductName': 'count'})
df1

,count,count
0,BIZHUB C451I W DF-713 45 PPM MFP,8604
1,BIZHUB C301I W DF-714 30 PPM MFP,5853
2,BIZHUB C361I W DF-714 36 PPM MFP,4875
3,BIZHUB C251I W DF-714 25 PPM MFP,4803
4,BIZHUB C450I 45 PPM COLOR MFP,4606
...,...,...
745,ACCURIOPRESS C7090E TAA COMPLIANT,1
746,REMOTE INSTLLTN PER PRNT ENBLMNT PCK,1
747,KIP 7170 SYSTEM INSTALLATION KMBS,1
748,PWRFILTER W INRUSH PROTECT 120V 15A,1


In [114]:
df1.to_csv('/Users/lubina.zaidi/Documents/Claims audit and fraud detection/ProductName_count.csv', index=False)


In [52]:
df.ClaimItemStatusReason.value_counts()

ClaimItemStatusReason
Completed                                             112121
Rejected due to KMAP Sale                               2346
Select For Audit                                        1532
Passed System Validation & Audit not Required           1161
Rejected                                                 246
Recalled                                                 175
Other                                                    155
Duplicate Claim                                            9
Submitted                                                  7
No Match Found                                             2
unit's ineligible as it's replacement & was unsold         1
Approved by Auditor                                        1
Name: count, dtype: int64

Once a claim has passed System Validation it goes to the Audit Process- approved - completed sattus
Audit process - edit and fix problems - Returned
Audit process - rejected 

Rejected due to KMAP Sale - KM specfic type of sale - claims submitted were not eligible for sales, so they were rejected
Completed - once claim a
returned. - The system validates that the Product ID and Product Unique ID match and someone else has claimed it.
Pending - claim is submitted and is waiting for approval after all teh required documents are submitted and has passed intial validation checks.
Draft - Claims in Draft Status may be Canceled by the submitter or claimant.Claims in Pending System Validation or Returned status may be recalled to Draft Status if information needs to be updated in the claim.  

If product does not have a Product Unique ID (serial number, VIN) to validate, then the claim will pass system validation automatically. 

In [53]:
df.ClaimItemStatus.value_counts()

ClaimItemStatus
Processed    114737
Cancelled     10443
Rejected       2750
Draft           782
Approved         74
Returned          9
Pending           9
Audit             1
Name: count, dtype: int64

In [54]:
df.IsSerializedProduct.value_counts()

IsSerializedProduct
Y    126317
Name: count, dtype: int64

In [55]:
128805 - 126317

2488

In [144]:
df.DealerName.value_counts()

DealerName
PACIFIC OFFICE AUTOMATION, INC.    26584
PERRY PRO TECH, INC.                7661
ALL COPY PRODUCTS, LLC              7298
DBA NOVATECH-MEMPHIS                6083
EDWARDS BUSINESS MACHINES, INC.     4534
                                   ...  
COPITEX BUSINESS MACHINES              1
PROCOPY, INC.                          1
O/C BUSINESS SYSTEMS                   1
STEAMROLLER COPIES, INC.               1
Southeast Sls                          1
Name: count, Length: 224, dtype: int64

Dates conversion to YYYY-MM-DD. num of days is calculated from teh invoice date - the product was sold and itemclaim date , when the claim was filed.

In [145]:


# Format selected date columns as YYYY-MM-DD strings
date_cols_to_format = ['InvoiceDate', 'ItemClaimedDate', 'ClaimItemStatusDate']
# STEP 2: Convert dates
for col in date_cols_to_format:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce').dt.strftime('%Y-%m-%d')

#df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')
#df['ItemClaimedDate'] = pd.to_datetime(df['ItemClaimedDate'], errors='coerce')
#df['ClaimSubmitionDate'] = pd.to_datetime(df['ClaimSubmitionDate'], errors='coerce')

print("\nSTEP 2: After converting dates")
print(df[['ClaimantName', 'InvoiceDate', 'ItemClaimedDate', 'ClaimSubmitionDate']].head())


# STEP 2: Calculate delay 


# compute days difference using temporary datetime conversion (safe if cols are strings)
invoicedate= pd.to_datetime(df['InvoiceDate'], errors='coerce')
itemclaimeddate = pd.to_datetime(df['ItemClaimedDate'], errors='coerce')
df['num_days'] = (itemclaimeddate - invoicedate).dt.days.round(0).astype("Int64")

print("\nSTEP 3: Delay calculation")
print(df[['ClaimantName', 'InvoiceDate', 'ItemClaimedDate', 'ClaimSubmitionDate', 'num_days']].head(3))





STEP 2: After converting dates
         ClaimantName InvoiceDate ItemClaimedDate      ClaimSubmitionDate
0       Sheila Mundey  2022-02-17      2022-02-18 2022-02-18 18:12:23.856
1  Christian Colasono  2021-08-10      2022-03-02 2022-03-02 14:48:57.054
2          Ryan Eades  2022-02-01      2022-02-24 2022-02-24 14:55:45.287
3          Admin User  2022-02-09      2022-02-16 2022-02-16 10:24:56.946
4       Sheila Mundey  2020-02-01      2022-02-16 2022-02-16 13:50:59.921

STEP 3: Delay calculation
         ClaimantName InvoiceDate ItemClaimedDate      ClaimSubmitionDate  \
0       Sheila Mundey  2022-02-17      2022-02-18 2022-02-18 18:12:23.856   
1  Christian Colasono  2021-08-10      2022-03-02 2022-03-02 14:48:57.054   
2          Ryan Eades  2022-02-01      2022-02-24 2022-02-24 14:55:45.287   

   num_days  
0         1  
1       204  
2        23  


In [146]:
df.head(5)

,ClaimID,InvoiceDate,ClaimDate,ClaimItemId,ItemClaimedDate,ItemDescription,ItemQuantity,ClaimItemStatusDate,ClaimantName,ClaimantName.1,...,ProductDescription,ProductManafacturer,SerialNumber,ClaimItemStatus,ClaimItemStatusReason,SalesTransactionID,SaleQuantity,SaleAmount,AuditComment,num_days
0,351322,2022-02-17,NaN,351322,2022-02-18,NaN,1,2022-02-18,Sheila Mundey,Sheila Mundey,...,NaN,NaN,a5yn017008872,Cancelled,NaN,NaN,NaN,NaN,NaN,1
1,351472,2021-08-10,NaN,351472,2022-03-02,NaN,1,2022-03-02,Christian Colasono,Christian Colasono,...,NaN,NaN,ACKN011001276,Processed,Select For Audit,386753.0,1.0,NaN,NaN,204
2,351483,2022-02-01,NaN,351483,2022-02-24,NaN,1,2022-02-24,Ryan Eades,Ryan Eades,...,NaN,NaN,AC57011020078,Processed,Select For Audit,351719.0,1.0,NaN,NaN,23
3,351182,2022-02-09,NaN,351182,2022-02-16,NaN,1,2022-02-16,Admin User,Admin User,...,NaN,NaN,12314344445,Cancelled,NaN,NaN,NaN,NaN,NaN,7
4,351187,2020-02-01,NaN,351187,2022-02-16,NaN,1,2022-02-16,Sheila Mundey,Sheila Mundey,...,NaN,NaN,124,Cancelled,NaN,NaN,NaN,NaN,NaN,746


In [147]:
df.columns

Index(['ClaimID', 'InvoiceDate', 'ClaimDate', 'ClaimItemId', 'ItemClaimedDate',
       'ItemDescription', 'ItemQuantity', 'ClaimItemStatusDate',
       'ClaimantName', 'ClaimantName.1', 'ClaimName', 'ProductSoldDate',
       'ClaimSubmitionDate', 'DealerName', 'ProductID', 'ProductStatus',
       'ProductName', 'IsSerializedProduct', 'ProductName.1',
       'ProductDescription', 'ProductManafacturer', 'SerialNumber',
       'ClaimItemStatus', 'ClaimItemStatusReason', 'SalesTransactionID',
       'SaleQuantity', 'SaleAmount', 'AuditComment', 'num_days'],
      dtype='str')

In [148]:
df['num_days_buckets'] = pd.cut(df['num_days'], bins=[-1, 0, 7, 14, 21, 30, 60, 90, 180, float('inf')],
                              labels=['0 days', '1-week', '2-week', '3-week', '4-week', '30-60 days', '60-90 days', '90-180 days', '180+ days']) 

Developing the Fraud and Anamoly flags

In [151]:

# fraud flag - if any of the below conditions are met, then flag as 1, else 0

pattern = r"Duplicate Claim|No Match Found|unit's ineligible as it's replacement & was unsold|Rejected"
df['fraud_flag'] = (
	df['ClaimItemStatusReason'].astype(str).str.contains(pattern, case=False, na=False)
	| df['ClaimItemStatus'].astype(str).str.contains(r"Rejected|Duplicate Claim|No Match Found", case=False, na=False)).astype(int)


 # Anomaly flag - if any of the below conditions are met, then flag as 1, else 0
df['anamoly_flag'] = (
    df['ClaimItemStatusReason'].astype(str).str.contains("Recalled", case=False, na=False)
    | df['ClaimItemStatus'].astype(str).str.contains(r"Cancelled|Returned|Draft|Pending", case=False, na=False)
    | df['num_days_buckets'].isin(['60-90 days', '90-180 days', '180+ days'])
).astype(int)

 Points to investigate  -  the product type 
 no of sales of the product in the last 6 months, if less than 5, then flag as anamoly
    # # qty 
    ## $ amount  
    ### avearge number of product sold. - of a particular item 
    ### claim avaerage sales per month of a particular product, if claim is more than 2 times the average sales per month, then flag as anamoly

In [152]:
df.head(3)

,ClaimID,InvoiceDate,ClaimDate,ClaimItemId,ItemClaimedDate,ItemDescription,ItemQuantity,ClaimItemStatusDate,ClaimantName,ClaimantName.1,...,ClaimItemStatus,ClaimItemStatusReason,SalesTransactionID,SaleQuantity,SaleAmount,AuditComment,num_days,num_days_buckets,fraud_flag,anamoly_flag
0,351322,2022-02-17,NaN,351322,2022-02-18,NaN,1,2022-02-18,Sheila Mundey,Sheila Mundey,...,Cancelled,NaN,NaN,NaN,NaN,NaN,1,1-week,0,1
1,351472,2021-08-10,NaN,351472,2022-03-02,NaN,1,2022-03-02,Christian Colasono,Christian Colasono,...,Processed,Select For Audit,386753.0,1.0,NaN,NaN,204,180+ days,0,1
2,351483,2022-02-01,NaN,351483,2022-02-24,NaN,1,2022-02-24,Ryan Eades,Ryan Eades,...,Processed,Select For Audit,351719.0,1.0,NaN,NaN,23,4-week,0,0


In [153]:
df.AuditComment.value_counts()

AuditComment
SN matches Dealer                                                                                360
SN matched Dealer                                                                                 20
SN Matches Dealer                                                                                  6
Confirmed Dealer SN                                                                                5
SN Matched Dealer                                                                                  2
Not an eligible product, no SN associated                                                          2
S/N matches rep and unit purchased by same dealership                                              2
S/N matches dealer                                                                                 2
sn matches dealer                                                                                  2
speak with DBM - Do we want to offer special points here?                     

In [165]:
df1 = df[df['fraud_flag'] == 1]
df1.head(10)


,ClaimID,InvoiceDate,ClaimDate,ClaimItemId,ItemClaimedDate,ItemDescription,ItemQuantity,ClaimItemStatusDate,ClaimantName,ClaimantName.1,...,ClaimItemStatus,ClaimItemStatusReason,SalesTransactionID,SaleQuantity,SaleAmount,AuditComment,num_days,num_days_buckets,fraud_flag,anamoly_flag
171,351704,2021-07-31,NaN,351704,2022-02-25,NaN,1,2022-02-25,Robert Thomas,Robert Thomas,...,Rejected,Other,NaN,NaN,NaN,NaN,209,180+ days,1,1
423,352212,2022-02-11,NaN,352212,2022-03-01,NaN,1,2022-03-01,Blair Gerber,Blair Gerber,...,Rejected,Other,NaN,NaN,NaN,NaN,18,3-week,1,0
463,352665,2022-02-16,NaN,352665,2022-03-04,NaN,1,2022-03-04,Blair Gerber,Blair Gerber,...,Rejected,Other,NaN,NaN,NaN,NaN,16,3-week,1,0
476,352286,2022-02-23,NaN,352286,2022-03-02,NaN,1,2022-03-02,Blair Gerber,Blair Gerber,...,Rejected,Other,NaN,NaN,NaN,NaN,7,1-week,1,0
502,352799,2022-02-21,NaN,352799,2022-03-07,NaN,1,2022-03-07,Blair Gerber,Blair Gerber,...,Rejected,Other,NaN,NaN,NaN,NaN,14,2-week,1,0
507,352662,2022-01-26,NaN,352662,2022-03-04,NaN,1,2022-03-04,Blair Gerber,Blair Gerber,...,Rejected,Other,NaN,NaN,NaN,NaN,37,30-60 days,1,0
510,352666,2022-02-04,NaN,352666,2022-03-04,NaN,1,2022-03-04,Blair Gerber,Blair Gerber,...,Rejected,Other,NaN,NaN,NaN,NaN,28,4-week,1,0
538,352669,2022-01-26,NaN,352669,2022-03-04,NaN,1,2022-03-04,Blair Gerber,Blair Gerber,...,Rejected,Other,NaN,NaN,NaN,NaN,37,30-60 days,1,0
548,352805,2022-02-02,NaN,352805,2022-03-07,NaN,1,2022-03-07,Blair Gerber,Blair Gerber,...,Rejected,Other,NaN,NaN,NaN,NaN,33,30-60 days,1,0
553,352798,2022-01-26,NaN,352798,2022-03-07,NaN,1,2022-03-07,Blair Gerber,Blair Gerber,...,Rejected,Other,NaN,NaN,NaN,NaN,40,30-60 days,1,0


Dealer-level product mix analysis

In [156]:


# ensure product/dealer names
_prod = df[['DealerName', 'ProductName','ClaimItemStatus','ClaimItemStatusReason']].fillna('Unknown')

# counts of claims per dealer-product
dealer_product_counts = (
    _prod
    .groupby(['DealerName', 'ProductName', 'ClaimItemStatus', 'ClaimItemStatusReason'])
    .size()
    .reset_index(name='count')
)

rows = []
for dealer, grp in dealer_product_counts.groupby('DealerName'):
    total = int(grp['count'].sum())
    counts = grp['count'].values
    unique_products = len(grp)
    # compute entropy from counts (use probabilities)
    if total > 0:
        probs = counts / total
        entropy_val = float(-np.sum(probs * np.log2(probs + 1e-12)))
    else:
        entropy_val = 0.0
    # compute HHI (Herfindahl-Hirschman Index) from counts (0-1 scale)
    hhi_val = float(np.sum(probs ** 2)) if total > 0 else 0.0
    # compute Gini coefficient (0-1 scale, where 0 = perfect equality, 1 = perfect inequality)
    sorted_counts = np.sort(counts)
    n = len(sorted_counts)
    gini_val = float((2 * np.sum(np.arange(1, n + 1) * sorted_counts)) / (n * np.sum(sorted_counts)) - (n + 1) / n) if total > 0 else 0.0
    top_share = float(grp['count'].max()) / total if total > 0 else 0.0

    top3 = (
        grp.sort_values('count', ascending=False)
        .head(3)
        .assign(share=lambda x: x['count'] / total * 100)
    )
    top3_str = "; ".join(
        [f"{r.ProductName} ({int(r.count)} | {r.share:.1f}%)" for r in top3.itertuples()]
    )

    rows.append({
        'DealerName': dealer,
        'total_claims': total,
        'unique_products': unique_products,
        'product_entropy': entropy_val,
        'product_hhi': hhi_val,
        'product_gini': gini_val,
        'top_product_share': top_share,
        'top_3_products': top3_str
    })

dealer_product_summary = pd.DataFrame(rows)

# sort and show
dealer_product_summary = dealer_product_summary.sort_values(
    ['unique_products', 'product_entropy'], ascending=[False, False]
).reset_index(drop=True)

# display top dealers by product variety and sample outputs
print("Top 20 dealers by unique products:")
display(dealer_product_summary.head(20))

print("\nTop 20 dealers by product entropy:")
display(dealer_product_summary.sort_values('product_entropy', ascending=False).head(20))

# create pivot table of top products for top dealers (for quick inspection)
top_dealers = dealer_product_summary.head(35)['DealerName'].tolist()
pivot_top = (
    dealer_product_counts[dealer_product_counts['DealerName'].isin(top_dealers)]
    .groupby(['DealerName', 'ProductName', 'ClaimItemStatus', 'ClaimItemStatusReason'])
    .agg({'count': 'sum'})
    .reset_index()
    .sort_values(['DealerName', 'count'], ascending=[True, False])
    .groupby('DealerName')
    .head(5)
    .pivot_table(index='DealerName', columns=[ 'ClaimItemStatusReason'], values='count', aggfunc='sum')
    .fillna(0)
)
display(pivot_top)

Top 20 dealers by unique products:


,DealerName,total_claims,unique_products,product_entropy,product_hhi,product_gini,top_product_share,top_3_products
0,"PACIFIC OFFICE AUTOMATION, INC.",26584,624,6.334368,0.026433,0.878631,0.066694,BIZHUB C750I PRINTER COPIER (1773 | 6.7%); BIZ...
1,DBA NOVATECH-MEMPHIS,6083,342,6.358933,0.026264,0.789708,0.079237,BIZHUB C451I W DF-713 45 PPM MFP (482 | 7.9%)...
2,"ALL COPY PRODUCTS, LLC",7298,267,5.997214,0.024994,0.812132,0.064401,BIZHUB C451I W DF-713 45 PPM MFP (470 | 6.4%)...
3,"EDWARDS BUSINESS MACHINES, INC.",4534,250,6.262939,0.021412,0.761112,0.054477,BIZHUB C451I W DF-713 45 PPM MFP (247 | 5.4%)...
4,"PERRY PRO TECH, INC.",7661,197,5.698736,0.030100,0.799042,0.068137,BIZHUB C450I 45 PPM COLOR MFP (522 | 6.8%); BI...
5,DBA FLEXTG - NORTHERN CA,3522,183,5.886595,0.028145,0.748777,0.083475,BIZHUB C551I W DF-713 55 PPM MFP (294 | 8.3%)...
6,DBA FISHER'S TECHNOLOGY,2386,178,5.791427,0.032644,0.748477,0.088852,BIZHUB C251I W DF-714 25 PPM MFP (212 | 8.9%)...
7,DBA ATLANTIC TOMORROWS OFFICE,2619,172,5.597103,0.035097,0.773096,0.082856,BIZHUB C451I W DF-713 45 PPM MFP (217 | 8.3%)...
8,DBA PROSOURCE,3967,168,5.768991,0.029461,0.750539,0.067557,BIZHUB C361I W DF-714 36 PPM MFP (268 | 6.8%)...
9,DBA FUNCTION4,1396,166,5.996588,0.027515,0.690544,0.073066,BIZHUB C301I W DF-714 30 PPM MFP (102 | 7.3%)...



Top 20 dealers by product entropy:


,DealerName,total_claims,unique_products,product_entropy,product_hhi,product_gini,top_product_share,top_3_products
1,DBA NOVATECH-MEMPHIS,6083,342,6.358933,0.026264,0.789708,0.079237,BIZHUB C451I W DF-713 45 PPM MFP (482 | 7.9%)...
0,"PACIFIC OFFICE AUTOMATION, INC.",26584,624,6.334368,0.026433,0.878631,0.066694,BIZHUB C750I PRINTER COPIER (1773 | 6.7%); BIZ...
3,"EDWARDS BUSINESS MACHINES, INC.",4534,250,6.262939,0.021412,0.761112,0.054477,BIZHUB C451I W DF-713 45 PPM MFP (247 | 5.4%)...
14,"FORD BUSINESS MACHINES, INC.",1128,149,6.043481,0.025272,0.652982,0.081560,FS-539 (92 | 8.2%); PC-416 CABINET (68 | 6.0%)...
2,"ALL COPY PRODUCTS, LLC",7298,267,5.997214,0.024994,0.812132,0.064401,BIZHUB C451I W DF-713 45 PPM MFP (470 | 6.4%)...
9,DBA FUNCTION4,1396,166,5.996588,0.027515,0.690544,0.073066,BIZHUB C301I W DF-714 30 PPM MFP (102 | 7.3%)...
31,DBA MMIT BUSINESS SOLUTIONS GROUP,221,91,5.965470,0.022870,0.443638,0.067873,BIZHUB C451I W DF-713 45 PPM MFP (15 | 6.8%);...
5,DBA FLEXTG - NORTHERN CA,3522,183,5.886595,0.028145,0.748777,0.083475,BIZHUB C551I W DF-713 55 PPM MFP (294 | 8.3%)...
15,"JAMES IMAGING SYSTEMS, INC.",1176,137,5.823233,0.031136,0.670155,0.102891,BIZHUB C451I W DF-713 45 PPM MFP (121 | 10.3%...
11,"UBEO, LLC",1305,158,5.820740,0.036024,0.695393,0.121839,BIZHUB C3320I - COLOR A4 AIO 35 PPM (159 | 12....


ClaimItemStatusReason,Completed,Unknown
DealerName,,
"ALL COPY PRODUCTS, LLC",1792.0,0.0
AN UBEO COMPANY,885.0,0.0
"BLUE TECHNOLOGIES, INC.",1002.0,0.0
"BRADEN BUSINESS SYSTEMS, INC.",420.0,0.0
"COPYPRO, INC.",256.0,0.0
"CPI TECHNOLOGIES, LLC",186.0,0.0
DBA ATLANTIC TOMORROWS OFFICE,855.0,0.0
DBA BUDGET DOCUMENT TECHNOLOGY,324.0,0.0
DBA CTWP,209.0,0.0


In [157]:
dealer_reason_counts = (
    df.groupby(['DealerName', 'ClaimItemStatusReason',  'fraud_flag', 'anamoly_flag'])
      .size()
      .reset_index(name='count')
      .sort_values('count', ascending= False)
)

dealer_reason_counts

,DealerName,ClaimItemStatusReason,fraud_flag,anamoly_flag,count
485,"PACIFIC OFFICE AUTOMATION, INC.",Completed,0,0,15948
505,"PERRY PRO TECH, INC.",Completed,0,0,6908
10,"ALL COPY PRODUCTS, LLC",Completed,0,0,6719
486,"PACIFIC OFFICE AUTOMATION, INC.",Completed,0,1,4192
267,DBA NOVATECH-MEMPHIS,Completed,0,0,3807
...,...,...,...,...,...
170,DBA DIGITAL SERVICE & SALES,Duplicate Claim,1,1,1
174,DBA EASTERN MANAGED PRINT NETWORK,Passed System Validation & Audit not Required,0,0,1
175,DBA EASTERN MANAGED PRINT NETWORK,Passed System Validation & Audit not Required,0,1,1
66,BUSINESS OFFICE SYSTEMS,Passed System Validation & Audit not Required,0,0,1


In [158]:
test = dealer_reason_counts[dealer_reason_counts['DealerName'] == "PACIFIC OFFICE AUTOMATION, INC."]
test

,DealerName,ClaimItemStatusReason,fraud_flag,anamoly_flag,count
485,"PACIFIC OFFICE AUTOMATION, INC.",Completed,0,0,15948
486,"PACIFIC OFFICE AUTOMATION, INC.",Completed,0,1,4192
496,"PACIFIC OFFICE AUTOMATION, INC.",Rejected due to KMAP Sale,1,0,915
498,"PACIFIC OFFICE AUTOMATION, INC.",Select For Audit,0,0,752
497,"PACIFIC OFFICE AUTOMATION, INC.",Rejected due to KMAP Sale,1,1,293
499,"PACIFIC OFFICE AUTOMATION, INC.",Select For Audit,0,1,209
491,"PACIFIC OFFICE AUTOMATION, INC.",Passed System Validation & Audit not Required,0,0,152
494,"PACIFIC OFFICE AUTOMATION, INC.",Rejected,1,0,116
493,"PACIFIC OFFICE AUTOMATION, INC.",Recalled,0,1,97
495,"PACIFIC OFFICE AUTOMATION, INC.",Rejected,1,1,58


In [159]:
test.groupby('ClaimItemStatusReason').agg({'count': 'sum', 
                                           'fraud_flag': 'sum', 
                                           'anamoly_flag': 'sum'}).sort_values('count', ascending=False)

,count,fraud_flag,anamoly_flag
ClaimItemStatusReason,,,
Completed,20143,1,1
Rejected due to KMAP Sale,1208,2,1
Select For Audit,961,0,1
Rejected,174,2,1
Passed System Validation & Audit not Required,171,0,1
Recalled,97,0,1
Other,41,2,1
Submitted,3,0,1
Duplicate Claim,2,1,1


In [160]:
totals = dealer_reason_counts[['count', 'fraud_flag', 'anamoly_flag']].sum()
display(totals)

count           117756
fraud_flag         143
anamoly_flag       279
dtype: int64

In [161]:
# filter out 'Completed' (case-insensitive)
dealer_reason_non_completed = dealer_reason_counts[
    dealer_reason_counts['ClaimItemStatusReason'].fillna('').str.strip().str.lower() != 'completed'
].reset_index(drop=True)

display(dealer_reason_non_completed)

,DealerName,ClaimItemStatusReason,fraud_flag,anamoly_flag,count
0,"PACIFIC OFFICE AUTOMATION, INC.",Rejected due to KMAP Sale,1,0,915
1,"PACIFIC OFFICE AUTOMATION, INC.",Select For Audit,0,0,752
2,"PACIFIC OFFICE AUTOMATION, INC.",Rejected due to KMAP Sale,1,1,293
3,DBA NOVATECH-MEMPHIS,Passed System Validation & Audit not Required,0,0,237
4,"PACIFIC OFFICE AUTOMATION, INC.",Select For Audit,0,1,209
...,...,...,...,...,...
255,DBA DIGITAL SERVICE & SALES,Duplicate Claim,1,1,1
256,DBA EASTERN MANAGED PRINT NETWORK,Passed System Validation & Audit not Required,0,0,1
257,DBA EASTERN MANAGED PRINT NETWORK,Passed System Validation & Audit not Required,0,1,1
258,BUSINESS OFFICE SYSTEMS,Passed System Validation & Audit not Required,0,0,1


In [ ]:
ZONE 3 BUSINESS SOLUTIONS, INC.
- Entropy: 2.76 (moderate diversity)
- HHI: 0.203 (relatively spread out)
- Gini: 0.519 (moderate inequality)
- Top product share: 36.6% (BIZHUB C251I dominates)

In [ ]:


# ============================================================
# SIMPLIFIED CLAIMS FRAUD DETECTION PIPELINE
# ============================================================


# ============================================================
# 1. PREPARE DATA
# ============================================================

# def prepare_claims_data(df):
#     """Clean and create basic features"""
#     work = df.copy()
    
#     # Convert dates
#     work['claim_date'] = pd.to_datetime(work['claim_date'], errors='coerce')
#     work['submission_date'] = pd.to_datetime(work['submission_date'], errors='coerce')
#     work['install_date'] = pd.to_datetime(work['install_date'], errors='coerce')
    
#     # Extract time features
#     work['submission_hour'] = work['submission_date'].dt.hour
#     work['submission_dow'] = work['submission_date'].dt.dayofweek
#     work['is_after_hours'] = ((work['submission_hour'] < 8) | (work['submission_hour'] > 18) | (work['submission_dow'] >= 5)).astype(int)
    
#     # Days between install and claim
#     work['days_to_claim'] = (work['claim_date'] - work['install_date']).dt.days
    
#     # Flag rejections
#    df['is_rejected'] = df['claim_status'].astype(str).str.lower().isin(['rejected', 'denied']).astype(int)
    
#     return work

# rename_map = {
#     "ClaimID": "claim_id",
#     "ClaimantName": "claimant_id",        # <-- use this as claimant identifier
#     "DealerName": "dealer_id",
#     "ClaimSubmitionDate": "submission_timestamp",
#     "ItemClaimedDate": "claim_date",
#     "InvoiceDate": "install_date",
#     "ProductID": "product_sku",
#     "SerialNumber": "serial_number",
#     "SaleAmount": "claim_amount",
#     "ItemQuantity": "quantity",
#     "ClaimItemStatus": "claim_status",
#     "ClaimantName": "claimant_name",
#     "DealerName": "dealer_name",
#     "ProductName": "product_name"
# }

# # apply only mappings for columns that exist in df
# rename_map = {k: v for k, v in rename_map.items() if k in df.columns}
# claims = df.rename(columns=rename_map).copy()


# 'ClaimID', 'InvoiceDate', 'ClaimDate', 'ClaimItemId', 'ItemClaimedDate',
#        'ItemDescription', 'ItemQuantity', 'ClaimItemStatusDate',
#        'ClaimantName', 'ClaimantName.1', 'ClaimName', 'ProductSoldDate',
#        'ClaimSubmitionDate', 'DealerName', 'ProductID', 'ProductStatus',
#        'ProductName', 'IsSerializedProduct', 'ProductName.1',
#        'ProductDescription', 'ProductManafacturer', 'SerialNumber',
#        'ClaimItemStatus', 'ClaimItemStatusReason', 'SalesTransactionID',
#        'SaleQuantity', 'SaleAmount', 'AuditComment', 'num_days',
#        'num_days_buckets', 'fraud_flag', 'anamoly_flag',
#       dtype='str'



claims = df.copy()
#df['is_rejected'] = df['claim_status'].astype(str).str.lower().isin(['rejected', 'denied']).astype(int)
    
# ============================================================
# 2. CLAIMANT-LEVEL FEATURES
# ============================================================

def build_claimant_features(df):
    """Create claimant behavior features"""
    
    claimant = df.groupby('ClaimantName').agg(
        total_claims=('ClaimID', 'count'),
       # total_amount=('claim_amount', 'sum'),
       # avg_amount=('claim_amount', 'mean'),
        #std_amount=('claim_amount', 'std'),
        num_dealers=('DealerName', 'nunique'),
        num_products=('ProductID', 'nunique'),
        fraud_flag_rate=('fraud_flag', 'mean'),
        anamoly_flag_rate=('anamoly_flag', 'mean'),
        num_days_buckets=('num_days_buckets', lambda x: (x >= 60) & (x < 90)),
        avg_days_to_claim=('num_days_buckets', 'mean')
    ).reset_index()
    
    # Fill NaN
    claimant = claimant.fillna(0)
    
    return claimant


# ============================================================
# 3. DEALER-LEVEL FEATURES
# ============================================================

def build_dealer_features(df):
    """Create dealer behavior features"""
    
    dealer = df.groupby('DealerName').agg(
        total_claims=('ClaimID', 'count'),
        #total_amount=('SaleAmount', 'sum'),
        #avg_amount=('SaleAmount', 'mean'),
        num_claimants=('ClaimantName', 'nunique'),
        num_products=('ProductID', 'nunique'),
        fraud_flag_rate=('fraud_flag', 'mean'),
        num_days_buckets=('num_days_buckets', lambda x: (x >= 60) & (x < 90)),
    ).reset_index()
    
    dealer = dealer.fillna(0)
    
    return dealer


# ============================================================
# 4. ANOMALY SCORING WITH ISOLATION FOREST
# ============================================================

def score_anomalies(features_df, contamination=0.05):
    """Score fraud risk using Isolation Forest"""
    
    result = features_df.copy()
    
    # Select numeric columns for modeling
    feature_cols = [col for col in result.columns if col != 'claimant_id' and col != 'dealer_id']
    
    X = result[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    
    # Scale data
    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(X)
    
    # Fit Isolation Forest
    model = IsolationForest(n_estimators=100, contamination=contamination, random_state=42)
    model.fit(X_scaled)
    
    # Get anomaly scores
    raw_score = -model.decision_function(X_scaled)
    
    # Convert to 0-100 scale
    mm = MinMaxScaler(feature_range=(0, 100))
    result['fraud_risk_score'] = mm.fit_transform(raw_score.reshape(-1, 1))
    
    # Risk segment
    result['risk_level'] = pd.cut(result['fraud_risk_score'], bins=[-1, 60, 80, 100], labels=['Low', 'Medium', 'High'])
    
    return result


# ============================================================
# 5. RUN FULL PIPELINE
# ============================================================

def run_pipeline(claims_df):
    """Run complete fraud detection"""
    
    # Prepare
    #clean_df = prepare_claims_data(claims_df)
    clean_df = df.copy()  # Assuming data is already prepared for this example
    
    # Build features
    claimant_features = build_claimant_features(clean_df)
    dealer_features = build_dealer_features(clean_df)
    
    # Score
    claimant_scored = score_anomalies(claimant_features, contamination=0.05)
    dealer_scored = score_anomalies(dealer_features, contamination=0.05)
    
    # Merge scores back to claims
    clean_df = clean_df.merge(claimant_scored[['ClaimantName', 'fraud_risk_score']], on='ClaimantName', how='left', suffixes=('', '_claimant'))
    clean_df = clean_df.merge(dealer_scored[['DealerName', 'fraud_risk_score']], on='DealerName', how='left', suffixes=('_claimant', '_dealer'))
    
    # Final score
    clean_df['final_score'] = (clean_df['fraud_risk_score_claimant'] * 0.6 + clean_df['fraud_risk_score_dealer'] * 0.4)
    
    # Sort by risk
    clean_df = clean_df.sort_values('final_score', ascending=False)
    
    return claimant_scored, dealer_scored, clean_df

In [163]:
len(claims_ranked)

128805

Steamlined Isolation forest algorithm : Isolation Forest is one of the best starting algorithms because it quickly surfaces unusual claimant and dealer behavior without requiring labeled fraud data

In [ ]:
# ============================================================================
# IMPORTS
# ============================================================================

import numpy as np
import pandas as pd

from sklearn.preprocessing import RobustScaler, MinMaxScaler
from sklearn.ensemble import IsolationForest


# ============================================================================
# CLAIMANT FEATURES
# ============================================================================

def build_claimant_features(df):

    claimant_features = (

        df.groupby('ClaimantName')

        .agg(

            total_claims=('ClaimID', 'count'),

            num_dealers=('DealerName', 'nunique'),

            num_products=('ProductID', 'nunique'),

            fraud_flag_count=('fraud_flag', 'sum'),

            fraud_flag_rate=('fraud_flag', 'mean'),

            anomaly_flag_count=('anamoly_flag', 'sum'),

            anomaly_flag_rate=('anamoly_flag', 'mean'),

            claims_over_90days=(
                'num_days',
                lambda x: (x >= 90).sum()
            ),

            avg_days_to_claim=('num_days', 'mean')

        ).reset_index()
)

    # -------------------------------------------------------
    # Normalized Rate Features
    # Helps reduce bias toward high-volume claimants
    # -------------------------------------------------------

    claimant_features['claims_over_90days_rate'] = (
        claimant_features['claims_over_90days']
        /claimant_features['total_claims'])

    claimant_features['products_per_claim'] = (
        claimant_features['num_products']
        /claimant_features['total_claims'])

    claimant_features['dealers_per_claim'] = (
        claimant_features['num_dealers']
        / claimant_features['total_claims'])

    return claimant_features.fillna(0)


# ============================================================================
# DEALER FEATURES
# ============================================================================

def build_dealer_features(df):

    dealer_features = (

        df.groupby('DealerName')

        .agg(

            total_claims=('ClaimID', 'count'),

            num_claimants=('ClaimantName', 'nunique'),

            num_products=('ProductID', 'nunique'),

            fraud_flag_count=('fraud_flag', 'sum'),

            fraud_flag_rate=('fraud_flag', 'mean'),

            anomaly_flag_count=('anamoly_flag', 'sum'),

            anomaly_flag_rate=('anamoly_flag', 'mean'),

            claims_over_90days=(
                'num_days',
                lambda x: (x >= 90).sum()
            )

        ).reset_index())

    # -------------------------------------------------------
    # Normalized Rates
    # -------------------------------------------------------

    dealer_features['claims_over_90days_rate'] = (
        dealer_features['claims_over_90days']
        /
        dealer_features['total_claims'])

    dealer_features['claimants_per_claim'] = (
        dealer_features['num_claimants']
        /
        dealer_features['total_claims'])

    dealer_features['products_per_claim'] = (
        dealer_features['num_products']
        /
        dealer_features['total_claims'])

    return dealer_features.fillna(0)


# ============================================================================
# ISOLATION FOREST SCORING
# ============================================================================

def score_anomalies(features_df, contamination=0.05):

    result = features_df.copy()

    # Numeric features only
    feature_cols = (
        result
        .select_dtypes(include=np.number)
        .columns
        .tolist())

    X = (
        result[feature_cols]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0))

    # Scale
    scaler = RobustScaler()

    X_scaled = scaler.fit_transform(X)

    # Build model
    model = IsolationForest(
        n_estimators=100,
        contamination=contamination,
        random_state=42)

    model.fit(X_scaled)

    # --------------------------------------------------------
    # TRUE Isolation Forest anomaly output
    # --------------------------------------------------------

    result['model_anomaly_flag'] = (model.predict(X_scaled) == -1).astype(int)

    # --------------------------------------------------------
    # Continuous anomaly score
    # --------------------------------------------------------

    raw_score = -model.score_samples(X_scaled)

    score_scaler = MinMaxScaler(feature_range=(0, 100))

    result['fraud_risk_score'] = (

        score_scaler

        .fit_transform(
            raw_score.reshape(-1, 1)
        )

        .flatten()

    )

    result['fraud_risk_score'] = (
        result['fraud_risk_score']
        .clip(0, 100))

    # --------------------------------------------------------
    # Risk Categories
    # --------------------------------------------------------

    result['risk_level'] = pd.cut(

        result['fraud_risk_score'],

        bins=[-np.inf, 60, 80, np.inf],

        labels=[
            'Low',
            'Medium',
            'High'])

    return result


# ============================================================================
# MAIN PIPELINE
# ============================================================================

def run_pipeline(df, contamination=0.05):

    clean_df = df.copy()

    # --------------------------------------------------------
    # Build Features
    # --------------------------------------------------------

    claimant_features = build_claimant_features(clean_df)

    dealer_features = build_dealer_features(clean_df)

    # --------------------------------------------------------
    # Score Claimants
    # --------------------------------------------------------

    claimant_scored = score_anomalies(claimant_features,contamination=contamination)

    # --------------------------------------------------------
    # Score Dealers
    # --------------------------------------------------------

    dealer_scored = score_anomalies(dealer_features,contamination=contamination
    )

    # --------------------------------------------------------
    # Rename
    # --------------------------------------------------------

    claimant_scored = claimant_scored.rename(
        columns={
            'fraud_risk_score':'fraud_risk_score_claimant',

            'risk_level': 'claimant_risk_level',

            'model_anomaly_flag': 'claimant_anomaly_flag'})

    dealer_scored = dealer_scored.rename(
        columns={
            'fraud_risk_score':'fraud_risk_score_dealer',

            'risk_level':'dealer_risk_level',

            'model_anomaly_flag': 'dealer_anomaly_flag'})

    # --------------------------------------------------------
    # Merge Claimant Risk
    # --------------------------------------------------------

    clean_df = clean_df.merge(

        claimant_scored[
            [
                'ClaimantName',
                'fraud_risk_score_claimant',
                'claimant_risk_level',
                'claimant_anomaly_flag'
            ]],on='ClaimantName',how='left')

    # --------------------------------------------------------
    # Merge Dealer Risk
    # --------------------------------------------------------

    clean_df = clean_df.merge(

        dealer_scored[
            [
                'DealerName',
                'fraud_risk_score_dealer',
                'dealer_risk_level',
                'dealer_anomaly_flag']],on='DealerName',how='left')

    # --------------------------------------------------------
    # Final Combined Score. - coul dbe 60 / 40 
    # --------------------------------------------------------

    clean_df['final_score'] = (

        clean_df['fraud_risk_score_claimant'] * 0.50

        +

        clean_df['fraud_risk_score_dealer'] * 0.50

    )

    clean_df['risk_level'] = pd.cut(

        clean_df['final_score'],

        bins=[-np.inf, 60, 80, np.inf],

        labels=[
            'Low',
            'Medium',
            'High'
        ]
)
    clean_df = clean_df.sort_values('final_score',ascending=False)

    return (claimant_scored,dealer_scored,clean_df)

# ============================================================================
# RUN MODEL
# ============================================================================

claimant_risk, dealer_risk, claims_ranked = run_pipeline(
    df,contamination=0.05)

# ============================================================================
# TOP CLAIMANTS
# ============================================================================

print("\nTOP 20 CLAIMANTS")

display(
    claimant_risk
    .sort_values(
        'fraud_risk_score_claimant',
        ascending=False)
    .head(20))

# ============================================================================
# TOP DEALERS
# ============================================================================

print("\nTOP 20 DEALERS")
display(
    dealer_risk
    .sort_values(
        'fraud_risk_score_dealer',
        ascending=False).head(20)
)

# ============================================================================
# TOP CLAIMS
# ============================================================================

print("\nTOP 20 CLAIMS")
display(
    claims_ranked[
        [
            'ClaimID',
            'ClaimantName',
            'DealerName',
            'fraud_risk_score_claimant',
            'fraud_risk_score_dealer',
            'final_score',
            'risk_level',
            'claimant_anomaly_flag',
            'dealer_anomaly_flag'
        ]
    ].head(20))

# ============================================================================
# CONTAMINATION TEST
# ============================================================================

print("\n\nCONTAMINATION COMPARISON")

claimant_features = build_claimant_features(df)

for c in [0.01, 0.03, 0.05, 0.10]:

    scored = score_anomalies(
        claimant_features,
        contamination=c
    )

    print("\n" + "=" * 60)
    print(f"CONTAMINATION = {c}")
    print("=" * 60)

    print("\nAnomalies flagged by Isolation Forest:")
    print(

        scored['model_anomaly_flag']

        .value_counts()

        .sort_index()

    )

    print(
        f"Percent flagged = "
        f"{100 * scored['model_anomaly_flag'].mean():.2f}%"
    )


TOP 20 CLAIMANTS


,ClaimantName,total_claims,num_dealers,num_products,fraud_flag_count,fraud_flag_rate,anomaly_flag_count,anomaly_flag_rate,claims_over_90days,avg_days_to_claim,claims_over_90days_rate,products_per_claim,dealers_per_claim,claimant_anomaly_flag,fraud_risk_score_claimant,claimant_risk_level
1275,Mark Fulk,750,1,53,74,0.098667,499,0.665333,154,71.734417,0.205333,0.070667,0.001333,1,100.000000,High
426,Clint Jeney,2039,1,35,2,0.000981,1302,0.638548,702,71.433566,0.344286,0.017165,0.000490,1,98.271125,High
1806,Terry Mulvaney,432,1,21,0,0.000000,357,0.826389,299,107.900232,0.69213,0.048611,0.002315,1,93.427029,High
1951,Yolanda Gan,343,1,40,0,0.000000,288,0.839650,261,87.190769,0.760933,0.116618,0.002915,1,90.162846,High
24,Adam Jack,532,1,46,68,0.127820,204,0.383459,128,59.988395,0.240602,0.086466,0.001880,1,90.061079,High
507,Daniel Reigh,145,1,38,0,0.000000,145,1.000000,145,474.234483,1.0,0.262069,0.006897,1,88.205351,High
972,John Park,1,1,1,1,1.000000,1,1.000000,1,112.0,1.0,1.000000,1.000000,1,87.616908,High
1276,Mark Goldsmith,2,1,2,2,1.000000,2,1.000000,2,341.0,1.0,1.000000,0.500000,1,86.967510,High
1303,Matt Hughes,131,1,35,44,0.335878,74,0.564885,62,156.603053,0.473282,0.267176,0.007634,1,84.615054,High
570,Dennis Allison,2264,1,73,0,0.000000,415,0.183304,66,41.304723,0.029152,0.032244,0.000442,1,83.894433,High



TOP 20 DEALERS


,DealerName,total_claims,num_claimants,num_products,fraud_flag_count,fraud_flag_rate,anomaly_flag_count,anomaly_flag_rate,claims_over_90days,claims_over_90days_rate,claimants_per_claim,products_per_claim,dealer_anomaly_flag,fraud_risk_score_dealer,dealer_risk_level
179,"PACIFIC OFFICE AUTOMATION, INC.",26584,303,314,1428,0.053717,8658,0.325685,3078,0.115784,0.011398,0.011812,1,100.000000,High
86,DBA NOVATECH-MEMPHIS,6083,102,204,83,0.013645,1832,0.301167,333,0.054743,0.016768,0.033536,1,65.866257,Medium
5,"ALL COPY PRODUCTS, LLC",7298,119,183,15,0.002055,557,0.076322,156,0.021376,0.016306,0.025075,1,56.652483,Low
65,DBA FLEXTG - NORTHERN CA,3522,40,103,118,0.033504,1108,0.314594,628,0.178308,0.011357,0.029245,1,55.531540,Low
34,COPITEX BUSINESS MACHINES,1,1,1,1,1.000000,0,0.000000,0,0.0,1.000000,1.000000,1,50.002821,Low
143,JOHNCO BUSINESS EQUIPMENT,1,1,1,1,1.000000,0,0.000000,0,0.0,1.000000,1.000000,1,50.002821,Low
94,DBA PROSOURCE,3967,31,107,104,0.026216,816,0.205697,414,0.104361,0.007814,0.026973,1,49.316251,Low
40,Commercial Print,2,1,2,0,0.000000,2,1.000000,2,1.0,0.500000,1.000000,1,48.458397,Low
114,"EDWARDS BUSINESS MACHINES, INC.",4534,51,154,168,0.037053,656,0.144685,176,0.038818,0.011248,0.033966,1,47.025512,Low
115,"EXECUTIVE COLOR SYSTEMS, INC.",801,2,63,0,0.000000,388,0.484395,273,0.340824,0.002497,0.078652,1,44.049223,Low



TOP 20 CLAIMS


,ClaimID,ClaimantName,DealerName,fraud_risk_score_claimant,fraud_risk_score_dealer,final_score,risk_level,claimant_anomaly_flag,dealer_anomaly_flag
99793,1825031,Mark Fulk,"PACIFIC OFFICE AUTOMATION, INC.",100.0,100.0,100.0,High,1,1
100012,1824979,Mark Fulk,"PACIFIC OFFICE AUTOMATION, INC.",100.0,100.0,100.0,High,1,1
46112,1129902,Mark Fulk,"PACIFIC OFFICE AUTOMATION, INC.",100.0,100.0,100.0,High,1,1
46111,1129884,Mark Fulk,"PACIFIC OFFICE AUTOMATION, INC.",100.0,100.0,100.0,High,1,1
46110,1129870,Mark Fulk,"PACIFIC OFFICE AUTOMATION, INC.",100.0,100.0,100.0,High,1,1
42837,1103357,Mark Fulk,"PACIFIC OFFICE AUTOMATION, INC.",100.0,100.0,100.0,High,1,1
9542,525002,Mark Fulk,"PACIFIC OFFICE AUTOMATION, INC.",100.0,100.0,100.0,High,1,1
9543,525007,Mark Fulk,"PACIFIC OFFICE AUTOMATION, INC.",100.0,100.0,100.0,High,1,1
42827,1103325,Mark Fulk,"PACIFIC OFFICE AUTOMATION, INC.",100.0,100.0,100.0,High,1,1
106253,1825012,Mark Fulk,"PACIFIC OFFICE AUTOMATION, INC.",100.0,100.0,100.0,High,1,1




CONTAMINATION COMPARISON

CONTAMINATION = 0.01

Anomalies flagged by Isolation Forest:
model_anomaly_flag
0    1949
1      20
Name: count, dtype: int64
Percent flagged = 1.02%

CONTAMINATION = 0.03

Anomalies flagged by Isolation Forest:
model_anomaly_flag
0    1909
1      60
Name: count, dtype: int64
Percent flagged = 3.05%

CONTAMINATION = 0.05

Anomalies flagged by Isolation Forest:
model_anomaly_flag
0    1870
1      99
Name: count, dtype: int64
Percent flagged = 5.03%

CONTAMINATION = 0.1

Anomalies flagged by Isolation Forest:
model_anomaly_flag
0    1772
1     197
Name: count, dtype: int64
Percent flagged = 10.01%


In [ ]:
# pattern = r"Duplicate Claim|No Match Found|unit's ineligible as it's replacement & was unsold|Rejected"
# df['fraud_flag'] = (
# 	df['ClaimItemStatusReason'].astype(str).str.contains(pattern, case=False, na=False)
# 	| df['ClaimItemStatus'].astype(str).str.contains(r"Rejected|Duplicate Claim|No Match Found", case=False, na=False)
# ).astype(int)
# df['anamoly_flag'] = (
#     df['ClaimItemStatusReason'].astype(str).str.contains("Recalled", case=False, na=False)
#     | df['ClaimItemStatus'].astype(str).str.contains(r"Cancelled|Returned|Draft|Pending", case=False, na=False)
#     | df['num_days_buckets'].isin(['60-90 days', '90-180 days', '180+ days'])
# ).astype(int)

In [ ]:
# ============================================================
# CLAIMS FRAUD DETECTION PIPELINE
# Motorcycle 1: Claimant-level statistical fraud detection
# Motorcycle 2: Dealer-level statistical fraud detection
#
# Purpose:
# This code creates explainable fraud risk scores using claims history.
# It does NOT require labeled fraud data.
#
# Main outputs:
# 1. claimant_scored      -> one row per claimant with risk score
# 2. dealer_scored        -> one row per dealer with risk score
# 3. review_queue         -> one row per claim, prioritized for audit review
#
# Why this approach:
# - Claims fraud is often rare and unlabeled.
# - Unsupervised anomaly detection is useful when we do not yet have confirmed
#   fraud / not-fraud labels.
# - The code combines business rules, claimant behavior patterns,
#   dealer benchmarking, fuzzy matching, and anomaly models.
# ============================================================


# ============================================================
# 1. IMPORT REQUIRED PACKAGES
# ============================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import RobustScaler, MinMaxScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_extraction.text import TfidfVectorizer


# ============================================================
# 2. ASSUMED INPUT DATA STRUCTURE
# ============================================================

# This pipeline assumes your input dataframe is called `claims`.
#
# Expected core columns:
#
# claim_id                Unique claim identifier
# claimant_id             Unique claimant/customer/person identifier
# dealer_id               Dealer associated with the claim
# claim_date              Date claim was made or processed
# install_date            Date product/service was installed
# submission_timestamp    Exact timestamp when claim was submitted
# product_sku             Product identifier
# serial_number           Serial number or product-specific identifier
# claim_amount            Dollar amount of claim
# quantity                Quantity claimed
# claim_status            Status such as Approved, Rejected, Denied
# claimant_name           Claimant name
# claimant_email          Claimant email
# claimant_phone          Claimant phone number
# claimant_address        Claimant address
# dealer_region           Dealer region / territory / market
# program_id              Program or incentive campaign identifier
#
# If your actual column names are different, rename them before running:
#
# claims = claims.rename(columns={
#     "your_claim_column": "claim_id",
#     "your_customer_column": "claimant_id",
#     ...
# })


# ============================================================
# 3. DATA PREPARATION
# ============================================================

def prepare_claims_data(claims: pd.DataFrame) -> pd.DataFrame:
    """
    Cleans raw claims data and creates basic date/time features.

    Why this is needed:
    Fraud patterns often show up in timing behavior:
    - submissions outside business hours
    - claims submitted in bursts
    - unusual install-to-claim lag
    - period-end submission spikes

    This function standardizes dates and creates foundation fields
    used later by both claimant-level and dealer-level models.
    """

    df = claims.copy()

    # Convert date columns into pandas datetime format.
    # errors="coerce" turns invalid dates into NaT instead of crashing.
    df["claim_date"] = pd.to_datetime(df["claim_date"], errors="coerce")
    df["install_date"] = pd.to_datetime(df["install_date"], errors="coerce")
    df["submission_timestamp"] = pd.to_datetime(
        df["submission_timestamp"],
        errors="coerce"
    )

    # Extract useful calendar fields from submission timestamp.
    # These help detect timing anomalies.
    df["submission_date"] = df["submission_timestamp"].dt.date
    df["submission_hour"] = df["submission_timestamp"].dt.hour
    df["submission_dayofweek"] = df["submission_timestamp"].dt.dayofweek

    # Month is useful for dashboarding and optional sell-in comparison.
    df["submission_month"] = (
        df["submission_timestamp"]
        .dt.to_period("M")
        .astype(str)
    )

    # Flag claims submitted outside a normal business window.
    # Why:
    # Unusual after-hours or weekend submission patterns may indicate automation,
    # batching, or behavior that differs from normal claimant/dealer behavior.
    df["is_after_hours"] = (
        (df["submission_hour"] < 8) |
        (df["submission_hour"] > 18) |
        (df["submission_dayofweek"] >= 5)
    ).astype(int)

    # Compute lag between install date and claim date.
    # Why:
    # Very short, very long, or highly inconsistent lag can be suspicious.
    df["install_to_claim_days"] = (
        df["claim_date"] - df["install_date"]
    ).dt.days

    # Create rejection indicator from claim status.
    # Why:
    # High rejection rates may suggest low-quality or potentially abusive claims.
    df["is_rejected"] = (
        df["claim_status"]
        .astype(str)
        .str.lower()
        .isin(["rejected", "denied", "invalid"])
    ).astype(int)

    # Standardize text columns.
    # Why:
    # Lowercasing and trimming makes grouping, matching, and fuzzy comparison
    # more consistent.
    string_cols = [
        "claimant_id",
        "dealer_id",
        "product_sku",
        "serial_number",
        "claimant_name",
        "claimant_email",
        "claimant_phone",
        "claimant_address",
        "dealer_region",
        "program_id"
    ]

    for col in string_cols:
        if col in df.columns:
            df[col] = (
                df[col]
                .astype(str)
                .str.strip()
                .str.lower()
            )

    return df


# ============================================================
# 4. UTILITY FUNCTIONS
# ============================================================

def robust_z_score(series: pd.Series) -> pd.Series:
    """
    Calculates a robust z-score using median and MAD.

    Why use robust z-score instead of regular z-score:
    Claims and fraud data are usually skewed.
    A few extreme values can distort mean and standard deviation.
    Median and MAD are more resistant to outliers.

    Used for:
    - Dealer peer benchmarking
    - Comparing dealers within the same region
    """

    median = series.median()
    mad = np.median(np.abs(series - median))

    # If MAD is zero, all values are likely the same.
    # Return zero so we do not divide by zero.
    if mad == 0:
        return pd.Series(np.zeros(len(series)), index=series.index)

    return 0.6745 * (series - median) / mad


def entropy_from_counts(counts: pd.Series) -> float:
    """
    Calculates entropy from category counts.

    Why entropy:
    Entropy measures diversity.
    Low entropy means repetitive behavior.
    High entropy means more varied behavior.

    Fraud examples:
    - Claimant always submits the same SKU
    - Dealer claims are concentrated around very few claimants
    - Claimant always uses same dealer/product combination

    These are not automatically fraud, but they are useful risk signals.
    """

    if counts.sum() == 0:
        return 0

    probs = counts / counts.sum()

    return -np.sum(probs * np.log2(probs + 1e-9))


def gini(array) -> float:
    """
    Calculates Gini coefficient.

    Why Gini:
    Gini measures inequality or concentration.
    For dealers, it helps answer:
    Are claims evenly distributed across many claimants,
    or concentrated among only a few claimants?

    Higher Gini means higher concentration risk.
    """

    array = np.array(array, dtype=float)

    if len(array) == 0:
        return 0

    if np.amin(array) < 0:
        array = array - np.amin(array)

    array = array + 1e-9
    array = np.sort(array)

    n = len(array)
    index = np.arange(1, n + 1)

    return (
        np.sum((2 * index - n - 1) * array) /
        (n * np.sum(array))
    )


def hhi_from_counts(counts: pd.Series) -> float:
    """
    Calculates Herfindahl-Hirschman Index, or HHI.

    Why HHI:
    HHI measures concentration.
    In fraud detection, it is useful for identifying dealers where a large
    portion of claims comes from a small number of claimants.

    Higher HHI means claims are more concentrated.
    """

    if counts.sum() == 0:
        return 0

    shares = counts / counts.sum()

    return np.sum(shares ** 2)


# ============================================================
# 5. MOTORCYCLE 1:
# CLAIMANT-LEVEL FEATURE ENGINEERING
# ============================================================

def build_claimant_features(
    df: pd.DataFrame,
    audit_threshold: float = None
) -> pd.DataFrame:
    """
    Creates claimant-level behavioral features.

    Motorcycle 1 goal:
    Detect unusual claimant behavior using claims history.

    Examples of fraud signals:
    - unusually high claim volume
    - claims submitted too frequently
    - claims repeatedly just below audit threshold
    - high rejection rate
    - after-hours activity
    - low product diversity
    - highly consistent timing
    """

    work = df.copy()

    # Sort by claimant and timestamp to calculate time between submissions.
    work = work.sort_values(["claimant_id", "submission_timestamp"])

    # Previous submission timestamp for each claimant.
    work["prev_submission_timestamp"] = (
        work.groupby("claimant_id")["submission_timestamp"]
        .shift(1)
    )

    # Time since prior claim, in hours.
    # Why:
    # Very short intervals or perfectly spaced intervals can indicate batching,
    # automation, or scripted submissions.
    work["hours_since_prior_submission"] = (
        work["submission_timestamp"] -
        work["prev_submission_timestamp"]
    ).dt.total_seconds() / 3600

    # Claims just below audit threshold.
    # Why:
    # If claimants learn that claims above a certain amount are audited,
    # fraudulent behavior may cluster just below the threshold.
    if audit_threshold is not None:
        work["near_below_audit_threshold"] = (
            (work["claim_amount"] >= audit_threshold * 0.90) &
            (work["claim_amount"] < audit_threshold)
        ).astype(int)
    else:
        work["near_below_audit_threshold"] = 0

    # Daily claim counts per claimant.
    daily = (
        work.groupby(["claimant_id", "submission_date"])
        .agg(daily_claim_count=("claim_id", "count"))
        .reset_index()
    )

    daily["submission_date"] = pd.to_datetime(daily["submission_date"])
    daily = daily.sort_values(["claimant_id", "submission_date"])

    # Rolling 7-day volume.
    # Why:
    # Velocity spikes are often more useful than lifetime totals.
    # A claimant may not look suspicious overall but may suddenly spike.
    daily["rolling_7d_claim_count"] = (
        daily.groupby("claimant_id")["daily_claim_count"]
        .transform(lambda x: x.rolling(7, min_periods=1).sum())
    )

    daily_agg = (
        daily.groupby("claimant_id")
        .agg(
            max_7d_claim_count=("rolling_7d_claim_count", "max"),
            avg_daily_claim_count=("daily_claim_count", "mean"),
            std_daily_claim_count=("daily_claim_count", "std")
        )
        .reset_index()
    )

    # Product entropy per claimant.
    # Why:
    # Low product diversity may indicate repetitive exploitation of one product.
    product_entropy = (
        work.groupby("claimant_id")["product_sku"]
        .apply(lambda x: entropy_from_counts(x.value_counts()))
        .reset_index(name="product_entropy")
    )

    # Dealer entropy per claimant.
    # Why:
    # A claimant using only one dealer may be normal,
    # but combined with other high-risk signals it may matter.
    dealer_entropy = (
        work.groupby("claimant_id")["dealer_id"]
        .apply(lambda x: entropy_from_counts(x.value_counts()))
        .reset_index(name="dealer_entropy")
    )

    # Aggregate claimant behavior.
    claimant_features = (
        work.groupby("claimant_id")
        .agg(
            claim_count=("claim_id", "count"),
            unique_dealers=("dealer_id", "nunique"),
            unique_products=("product_sku", "nunique"),
            unique_programs=("program_id", "nunique"),

            total_claim_amount=("claim_amount", "sum"),
            avg_claim_amount=("claim_amount", "mean"),
            std_claim_amount=("claim_amount", "std"),

            total_quantity=("quantity", "sum"),
            avg_quantity=("quantity", "mean"),

            rejection_rate=("is_rejected", "mean"),
            after_hours_rate=("is_after_hours", "mean"),
            near_threshold_rate=("near_below_audit_threshold", "mean"),

            avg_install_to_claim_days=("install_to_claim_days", "mean"),
            std_install_to_claim_days=("install_to_claim_days", "std"),

            avg_hours_between_claims=("hours_since_prior_submission", "mean"),
            std_hours_between_claims=("hours_since_prior_submission", "std"),
            min_hours_between_claims=("hours_since_prior_submission", "min"),

            first_submission=("submission_timestamp", "min"),
            last_submission=("submission_timestamp", "max")
        )
        .reset_index()
    )

    # Number of days between first and last submission.
    # Why:
    # Helps distinguish a long-active claimant from a short burst claimant.
    claimant_features["active_days"] = (
        claimant_features["last_submission"] -
        claimant_features["first_submission"]
    ).dt.days + 1

    # Claim velocity.
    # Why:
    # Normalizes claim count by active period.
    claimant_features["claims_per_active_day"] = (
        claimant_features["claim_count"] /
        claimant_features["active_days"].replace(0, 1)
    )

    # Coefficient of variation for timing.
    # Why:
    # Very low variation can indicate mechanical or automated submissions.
    claimant_features["timing_cv"] = (
        claimant_features["std_hours_between_claims"] /
        claimant_features["avg_hours_between_claims"].replace(0, np.nan)
    )

    # Coefficient of variation for claim amount.
    # Why:
    # Identical or near-identical claim amounts repeatedly can be suspicious.
    claimant_features["amount_cv"] = (
        claimant_features["std_claim_amount"] /
        claimant_features["avg_claim_amount"].replace(0, np.nan)
    )

    # Merge all claimant feature blocks.
    claimant_features = claimant_features.merge(
        daily_agg,
        on="claimant_id",
        how="left"
    )

    claimant_features = claimant_features.merge(
        product_entropy,
        on="claimant_id",
        how="left"
    )

    claimant_features = claimant_features.merge(
        dealer_entropy,
        on="claimant_id",
        how="left"
    )

    # Fill numeric missing values.
    numeric_cols = claimant_features.select_dtypes(include=[np.number]).columns
    claimant_features[numeric_cols] = claimant_features[numeric_cols].fillna(0)

    return claimant_features


# ============================================================
# 6. MOTORCYCLE 1:
# SHARED IDENTITY FEATURES
# ============================================================

def add_shared_identity_features(
    df: pd.DataFrame,
    claimant_features: pd.DataFrame
) -> pd.DataFrame:
    """
    Adds shared identity signals.

    Why this matters:
    Fraud may be split across multiple claimant IDs that share the same:
    - email
    - phone
    - address

    This can indicate duplicate identities, household-level abuse,
    synthetic claimant creation, or coordinated behavior.
    """

    work = df.copy()
    result = claimant_features.copy()

    identity_cols = [
        "claimant_email",
        "claimant_phone",
        "claimant_address"
    ]

    for col in identity_cols:
        if col not in work.columns:
            continue

        # Count how many unique claimant IDs share the same identity attribute.
        shared_map = (
            work.groupby(col)["claimant_id"]
            .nunique()
            .reset_index(name=f"{col}_shared_claimant_count")
        )

        work = work.merge(shared_map, on=col, how="left")

        # Roll up shared identity count to claimant level.
        claimant_shared = (
            work.groupby("claimant_id")[f"{col}_shared_claimant_count"]
            .max()
            .reset_index()
        )

        result = result.merge(
            claimant_shared,
            on="claimant_id",
            how="left"
        )

    numeric_cols = result.select_dtypes(include=[np.number]).columns
    result[numeric_cols] = result[numeric_cols].fillna(0)

    return result


# ============================================================
# 7. MOTORCYCLE 1:
# FUZZY DUPLICATE DETECTION
# ============================================================

def fuzzy_duplicate_score(
    df: pd.DataFrame,
    text_col: str,
    id_col: str = "claim_id",
    min_similarity: float = 0.90
) -> pd.DataFrame:
    """
    Finds near-duplicate text values using character-level TF-IDF.

    Why this is useful:
    Exact duplicate rules catch identical values only.
    Fraudulent or duplicate claims may use small variations, such as:
    - serial number typo
    - name spelling variation
    - spacing/punctuation differences

    Why TF-IDF with character n-grams:
    Character n-grams are good for fuzzy text similarity because they detect
    partial overlaps and small edits without requiring exact matches.
    """

    work = df[[id_col, text_col]].copy()

    work[text_col] = (
        work[text_col]
        .fillna("")
        .astype(str)
        .str.lower()
        .str.strip()
    )

    nonblank = work[work[text_col] != ""].copy()

    # If fewer than two records exist, no comparison is possible.
    if len(nonblank) < 2:
        work[f"{text_col}_max_similarity"] = 0
        work[f"{text_col}_near_duplicate_flag"] = 0

        return work[
            [
                id_col,
                f"{text_col}_max_similarity",
                f"{text_col}_near_duplicate_flag"
            ]
        ]

    # Character n-gram TF-IDF.
    # analyzer="char_wb" uses character sequences inside word boundaries.
    vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=1
    )

    X = vectorizer.fit_transform(nonblank[text_col])

    # Nearest neighbor finds the most similar record for each row.
    # Cosine distance is used because TF-IDF vectors are sparse text vectors.
    nn = NearestNeighbors(
        n_neighbors=min(2, len(nonblank)),
        metric="cosine"
    )

    nn.fit(X)

    distances, indices = nn.kneighbors(X)

    # First nearest neighbor is usually the record itself.
    # Second nearest neighbor is the closest other record.
    if distances.shape[1] > 1:
        nearest_distance = distances[:, 1]
    else:
        nearest_distance = distances[:, 0]

    max_similarity = 1 - nearest_distance

    nonblank[f"{text_col}_max_similarity"] = max_similarity

    nonblank[f"{text_col}_near_duplicate_flag"] = (
        nonblank[f"{text_col}_max_similarity"] >= min_similarity
    ).astype(int)

    output = work[[id_col]].merge(
        nonblank[
            [
                id_col,
                f"{text_col}_max_similarity",
                f"{text_col}_near_duplicate_flag"
            ]
        ],
        on=id_col,
        how="left"
    )

    output[f"{text_col}_max_similarity"] = (
        output[f"{text_col}_max_similarity"]
        .fillna(0)
    )

    output[f"{text_col}_near_duplicate_flag"] = (
        output[f"{text_col}_near_duplicate_flag"]
        .fillna(0)
    )

    return output


# ============================================================
# 8. MOTORCYCLE 1:
# CLAIMANT ANOMALY SCORING
# ============================================================

def score_claimant_anomalies(
    claimant_features: pd.DataFrame,
    contamination: float = 0.05
) -> pd.DataFrame:
    """
    Scores claimant-level fraud risk using Isolation Forest.

    Why Isolation Forest:
    - Works without labeled fraud data.
    - Designed to isolate unusual observations.
    - Good first model for rare anomaly detection.
    - Handles many numeric behavioral features well.

    contamination:
    Expected fraction of unusual claimants.
    Example:
    contamination=0.05 means the model expects about 5% of claimants
    to look anomalous.
    """

    result = claimant_features.copy()

    feature_cols = [
        "claim_count",
        "unique_dealers",
        "unique_products",
        "unique_programs",
        "total_claim_amount",
        "avg_claim_amount",
        "total_quantity",
        "avg_quantity",
        "rejection_rate",
        "after_hours_rate",
        "near_threshold_rate",
        "avg_install_to_claim_days",
        "std_install_to_claim_days",
        "avg_hours_between_claims",
        "std_hours_between_claims",
        "min_hours_between_claims",
        "claims_per_active_day",
        "timing_cv",
        "amount_cv",
        "max_7d_claim_count",
        "avg_daily_claim_count",
        "std_daily_claim_count",
        "product_entropy",
        "dealer_entropy"
    ]

    # Optional features that may or may not exist depending on available data.
    optional_cols = [
        "claimant_email_shared_claimant_count",
        "claimant_phone_shared_claimant_count",
        "claimant_address_shared_claimant_count",
        "max_serial_similarity",
        "serial_near_duplicate_rate",
        "max_name_similarity",
        "name_near_duplicate_rate"
    ]

    feature_cols = [
        c for c in feature_cols + optional_cols
        if c in result.columns
    ]

    # Replace infinite/missing values.
    X = (
        result[feature_cols]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    # RobustScaler is used because claims data is usually skewed.
    # It scales using median and quartiles instead of mean/std.
    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(X)

    model = IsolationForest(
        n_estimators=300,
        contamination=contamination,
        random_state=42
    )

    model.fit(X_scaled)

    # IsolationForest decision_function:
    # lower values mean more anomalous.
    # Multiplying by -1 makes higher values mean higher risk.
    raw_score = -model.decision_function(X_scaled)

    result["claimant_anomaly_raw_score"] = raw_score

    # Convert raw score to 0-100 for business interpretation.
    mm = MinMaxScaler(feature_range=(0, 100))

    result["claimant_anomaly_score"] = mm.fit_transform(
        raw_score.reshape(-1, 1)
    )

    # Segment risk for dashboards and reviewer queues.
    result["claimant_risk_segment"] = pd.cut(
        result["claimant_anomaly_score"],
        bins=[-1, 60, 80, 100],
        labels=["Low", "Medium", "High"]
    )

    return result


# ============================================================
# 9. MOTORCYCLE 1:
# CLAIMANT RISK DRIVER EXPLANATIONS
# ============================================================

def add_claimant_risk_drivers(
    claimant_scored: pd.DataFrame
) -> pd.DataFrame:
    """
    Adds human-readable reasons for claimant risk.

    Why this matters:
    Investigators and business users need to know WHY a claimant is risky.
    A black-box score alone is not enough for an audit workflow.

    This function ranks available driver features by percentile and returns
    the top three reason labels.
    """

    df = claimant_scored.copy()

    driver_rules = {
        "High claim velocity": "claims_per_active_day",
        "High rejection rate": "rejection_rate",
        "High after-hours activity": "after_hours_rate",
        "Claims near audit threshold": "near_threshold_rate",
        "High 7-day volume spike": "max_7d_claim_count",
        "Low product diversity": "product_entropy",
        "Low timing variation": "timing_cv",
        "Shared phone across claimants": "claimant_phone_shared_claimant_count",
        "Shared email across claimants": "claimant_email_shared_claimant_count",
        "Shared address across claimants": "claimant_address_shared_claimant_count",
        "Near-duplicate serials": "serial_near_duplicate_rate",
        "Near-duplicate names": "name_near_duplicate_rate"
    }

    percentile_cols = []

    for label, col in driver_rules.items():
        if col in df.columns:
            pct_col = f"{col}_percentile"

            # Percentile rank shows how extreme each claimant is
            # relative to other claimants.
            df[pct_col] = df[col].rank(pct=True)

            percentile_cols.append((label, pct_col))

    def top_drivers(row):
        scored = []

        for label, pct_col in percentile_cols:
            value = row[pct_col]

            if pd.notna(value):
                scored.append((label, value))

        scored = sorted(scored, key=lambda x: x[1], reverse=True)

        return "; ".join([x[0] for x in scored[:3]])

    df["top_claimant_risk_drivers"] = df.apply(top_drivers, axis=1)

    return df


# ============================================================
# 10. MOTORCYCLE 2:
# DEALER-LEVEL FEATURE ENGINEERING
# ============================================================

def build_dealer_features(
    df: pd.DataFrame,
    claimant_scored: pd.DataFrame = None
) -> pd.DataFrame:
    """
    Creates dealer-level behavioral and benchmarking features.

    Motorcycle 2 goal:
    Detect unusual dealer behavior using claims history.

    Dealer-level fraud signals:
    - dealer has unusually high claim volume
    - dealer claims are concentrated among few claimants
    - dealer has many high-risk claimants
    - dealer has temporal clustering
    - dealer has unusual rejection or after-hours patterns
    """

    work = df.copy()

    # Attach claimant risk to each claim.
    # Why:
    # Dealer risk should reflect the riskiness of claimants associated
    # with that dealer.
    if claimant_scored is not None and "claimant_anomaly_score" in claimant_scored.columns:
        work = work.merge(
            claimant_scored[
                [
                    "claimant_id",
                    "claimant_anomaly_score",
                    "claimant_risk_segment"
                ]
            ],
            on="claimant_id",
            how="left"
        )
    else:
        work["claimant_anomaly_score"] = 0
        work["claimant_risk_segment"] = "Unknown"

    # Dealer product entropy.
    # Why:
    # Low diversity may indicate repeated exploitation of the same SKU.
    dealer_product_entropy = (
        work.groupby("dealer_id")["product_sku"]
        .apply(lambda x: entropy_from_counts(x.value_counts()))
        .reset_index(name="dealer_product_entropy")
    )

    # Dealer claimant entropy.
    # Why:
    # Low claimant diversity may indicate concentration among few claimants.
    dealer_claimant_entropy = (
        work.groupby("dealer_id")["claimant_id"]
        .apply(lambda x: entropy_from_counts(x.value_counts()))
        .reset_index(name="dealer_claimant_entropy")
    )

    # Concentration metrics by dealer.
    concentration_rows = []

    for dealer_id, group in work.groupby("dealer_id"):
        claimant_counts = group["claimant_id"].value_counts()

        concentration_rows.append({
            "dealer_id": dealer_id,

            # HHI captures concentration.
            "claimant_hhi": hhi_from_counts(claimant_counts),

            # Gini captures inequality across claimants.
            "claimant_gini": gini(claimant_counts.values),

            # Top claimant share tells us how dependent the dealer is
            # on the highest-volume claimant.
            "top_claimant_share": (
                claimant_counts.max() / claimant_counts.sum()
                if claimant_counts.sum() > 0 else 0
            )
        })

    concentration = pd.DataFrame(concentration_rows)

    # Temporal clustering: number of claims submitted by dealer
    # in the same date/hour bucket.
    # Why:
    # Many claims at the same timestamp/hour can indicate batching.
    same_hour = (
        work.groupby(["dealer_id", "submission_date", "submission_hour"])
        .agg(claims_same_hour=("claim_id", "count"))
        .reset_index()
    )

    temporal = (
        same_hour.groupby("dealer_id")
        .agg(
            max_claims_same_hour=("claims_same_hour", "max"),
            avg_claims_same_hour=("claims_same_hour", "mean")
        )
        .reset_index()
    )

    # Dealer daily volume for rolling spike detection.
    dealer_daily = (
        work.groupby(["dealer_id", "submission_date"])
        .agg(daily_claim_count=("claim_id", "count"))
        .reset_index()
    )

    dealer_daily["submission_date"] = pd.to_datetime(
        dealer_daily["submission_date"]
    )

    dealer_daily = dealer_daily.sort_values(
        ["dealer_id", "submission_date"]
    )

    # Rolling 7-day claim volume per dealer.
    # Why:
    # Detects bursts and short-term spikes.
    dealer_daily["rolling_7d_claim_count"] = (
        dealer_daily.groupby("dealer_id")["daily_claim_count"]
        .transform(lambda x: x.rolling(7, min_periods=1).sum())
    )

    dealer_daily_features = (
        dealer_daily.groupby("dealer_id")
        .agg(
            max_dealer_7d_claim_count=("rolling_7d_claim_count", "max"),
            avg_dealer_daily_claim_count=("daily_claim_count", "mean"),
            std_dealer_daily_claim_count=("daily_claim_count", "std")
        )
        .reset_index()
    )

    # Core dealer-level aggregates.
    dealer_features = (
        work.groupby("dealer_id")
        .agg(
            dealer_claim_count=("claim_id", "count"),
            unique_claimants=("claimant_id", "nunique"),
            unique_products=("product_sku", "nunique"),
            unique_programs=("program_id", "nunique"),

            total_claim_amount=("claim_amount", "sum"),
            avg_claim_amount=("claim_amount", "mean"),

            total_quantity=("quantity", "sum"),
            avg_quantity=("quantity", "mean"),

            rejection_rate=("is_rejected", "mean"),
            after_hours_rate=("is_after_hours", "mean"),

            avg_install_to_claim_days=("install_to_claim_days", "mean"),
            std_install_to_claim_days=("install_to_claim_days", "std"),

            avg_claimant_risk_score=("claimant_anomaly_score", "mean"),
            max_claimant_risk_score=("claimant_anomaly_score", "max"),

            high_risk_claimant_count=(
                "claimant_risk_segment",
                lambda x: (x.astype(str) == "High").sum()
            ),

            dealer_region=("dealer_region", "first"),

            first_submission=("submission_timestamp", "min"),
            last_submission=("submission_timestamp", "max")
        )
        .reset_index()
    )

    # Dealer active period.
    dealer_features["active_days"] = (
        dealer_features["last_submission"] -
        dealer_features["first_submission"]
    ).dt.days + 1

    # Dealer claim velocity.
    dealer_features["claims_per_active_day"] = (
        dealer_features["dealer_claim_count"] /
        dealer_features["active_days"].replace(0, 1)
    )

    # Claimant-to-claim ratio.
    # Why:
    # Helps identify whether claims are spread across many claimants
    # or concentrated in a few.
    dealer_features["claimants_per_claim"] = (
        dealer_features["unique_claimants"] /
        dealer_features["dealer_claim_count"].replace(0, 1)
    )

    # Rate of high-risk claimants associated with dealer.
    dealer_features["high_risk_claimant_rate"] = (
        dealer_features["high_risk_claimant_count"] /
        dealer_features["unique_claimants"].replace(0, 1)
    )

    # Merge dealer feature blocks.
    dealer_features = dealer_features.merge(
        dealer_product_entropy,
        on="dealer_id",
        how="left"
    )

    dealer_features = dealer_features.merge(
        dealer_claimant_entropy,
        on="dealer_id",
        how="left"
    )

    dealer_features = dealer_features.merge(
        concentration,
        on="dealer_id",
        how="left"
    )

    dealer_features = dealer_features.merge(
        temporal,
        on="dealer_id",
        how="left"
    )

    dealer_features = dealer_features.merge(
        dealer_daily_features,
        on="dealer_id",
        how="left"
    )

    numeric_cols = dealer_features.select_dtypes(include=[np.number]).columns
    dealer_features[numeric_cols] = dealer_features[numeric_cols].fillna(0)

    return dealer_features


# ============================================================
# 11. MOTORCYCLE 2:
# DEALER PEER BENCHMARKING
# ============================================================

def add_dealer_peer_benchmarks(
    dealer_features: pd.DataFrame
) -> pd.DataFrame:
    """
    Compares each dealer against similar dealers in the same region.

    Why peer benchmarking:
    A dealer may look large overall but be normal for its region.
    Region-based benchmarking helps reduce false positives by comparing
    dealers to a more relevant peer group.

    This creates:
    - region median
    - ratio vs region median
    - robust z-score within region
    """

    df = dealer_features.copy()

    benchmark_cols = [
        "dealer_claim_count",
        "total_claim_amount",
        "claims_per_active_day",
        "rejection_rate",
        "after_hours_rate",
        "avg_claimant_risk_score",
        "claimant_hhi",
        "claimant_gini",
        "top_claimant_share",
        "max_dealer_7d_claim_count"
    ]

    for col in benchmark_cols:
        if col not in df.columns:
            continue

        region_median_col = f"region_median_{col}"
        ratio_col = f"{col}_vs_region_median"
        z_col = f"{col}_region_robust_z"

        # Median behavior of dealers in same region.
        df[region_median_col] = (
            df.groupby("dealer_region")[col]
            .transform("median")
        )

        # Ratio to peer median.
        df[ratio_col] = (
            df[col] /
            df[region_median_col].replace(0, np.nan)
        )

        # Robust z-score within region.
        df[z_col] = (
            df.groupby("dealer_region")[col]
            .transform(robust_z_score)
        )

    df = df.replace([np.inf, -np.inf], np.nan).fillna(0)

    return df


# ============================================================
# 12. OPTIONAL MOTORCYCLE 2:
# SELL-IN VS CLAIMS RATIO
# ============================================================

def add_sell_in_claim_ratio(
    claims_df: pd.DataFrame,
    dealer_features: pd.DataFrame,
    sell_in: pd.DataFrame
) -> pd.DataFrame:
    """
    Optional feature block if sell-in / wholesale data is available.

    Expected sell_in columns:
    - dealer_id
    - month
    - sell_in_units
    - sell_in_amount

    Why this matters:
    If claims grow much faster than dealer inventory or wholesale volume,
    it may indicate suspicious claim behavior.

    This feature is especially useful when sell-in data becomes available.
    """

    claims_monthly = claims_df.copy()

    claims_monthly["month"] = (
        claims_monthly["submission_timestamp"]
        .dt.to_period("M")
        .astype(str)
    )

    claims_monthly = (
        claims_monthly.groupby(["dealer_id", "month"])
        .agg(
            monthly_claim_count=("claim_id", "count"),
            monthly_claim_quantity=("quantity", "sum"),
            monthly_claim_amount=("claim_amount", "sum")
        )
        .reset_index()
    )

    sell = sell_in.copy()
    sell["month"] = sell["month"].astype(str)

    merged = claims_monthly.merge(
        sell,
        on=["dealer_id", "month"],
        how="left"
    )

    # Claims-to-sell-in ratio.
    # Why:
    # A very high ratio suggests claims are high relative to inventory flow.
    merged["claims_to_sell_in_units_ratio"] = (
        merged["monthly_claim_quantity"] /
        merged["sell_in_units"].replace(0, np.nan)
    )

    merged["claims_to_sell_in_amount_ratio"] = (
        merged["monthly_claim_amount"] /
        merged["sell_in_amount"].replace(0, np.nan)
    )

    dealer_sell_features = (
        merged.groupby("dealer_id")
        .agg(
            avg_claims_to_sell_in_units_ratio=(
                "claims_to_sell_in_units_ratio",
                "mean"
            ),
            max_claims_to_sell_in_units_ratio=(
                "claims_to_sell_in_units_ratio",
                "max"
            ),
            avg_claims_to_sell_in_amount_ratio=(
                "claims_to_sell_in_amount_ratio",
                "mean"
            ),
            max_claims_to_sell_in_amount_ratio=(
                "claims_to_sell_in_amount_ratio",
                "max"
            )
        )
        .reset_index()
    )

    result = dealer_features.merge(
        dealer_sell_features,
        on="dealer_id",
        how="left"
    )

    result = result.replace([np.inf, -np.inf], np.nan).fillna(0)

    return result


# ============================================================
# 13. MOTORCYCLE 2:
# DEALER ANOMALY SCORING
# ============================================================

def score_dealer_anomalies(
    dealer_features: pd.DataFrame,
    contamination: float = 0.05
) -> pd.DataFrame:
    """
    Scores dealer-level fraud risk using Isolation Forest.

    Why Isolation Forest:
    Dealer fraud patterns may be rare and unlabeled.
    Isolation Forest can detect dealers that behave differently from the
    rest of the population based on multiple features.
    """

    result = dealer_features.copy()

    feature_cols = [
        "dealer_claim_count",
        "unique_claimants",
        "unique_products",
        "unique_programs",
        "total_claim_amount",
        "avg_claim_amount",
        "total_quantity",
        "avg_quantity",
        "rejection_rate",
        "after_hours_rate",
        "avg_install_to_claim_days",
        "std_install_to_claim_days",
        "avg_claimant_risk_score",
        "max_claimant_risk_score",
        "high_risk_claimant_count",
        "high_risk_claimant_rate",
        "claims_per_active_day",
        "claimants_per_claim",
        "dealer_product_entropy",
        "dealer_claimant_entropy",
        "claimant_hhi",
        "claimant_gini",
        "top_claimant_share",
        "max_claims_same_hour",
        "avg_claims_same_hour",
        "max_dealer_7d_claim_count",
        "avg_dealer_daily_claim_count",
        "std_dealer_daily_claim_count"
    ]

    optional_cols = [
        "avg_claims_to_sell_in_units_ratio",
        "max_claims_to_sell_in_units_ratio",
        "avg_claims_to_sell_in_amount_ratio",
        "max_claims_to_sell_in_amount_ratio"
    ]

    # Add all region benchmark robust z-score columns.
    benchmark_cols = [
        c for c in result.columns
        if c.endswith("_region_robust_z")
    ]

    feature_cols = [
        c for c in feature_cols + optional_cols + benchmark_cols
        if c in result.columns
    ]

    X = (
        result[feature_cols]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(X)

    model = IsolationForest(
        n_estimators=300,
        contamination=contamination,
        random_state=42
    )

    model.fit(X_scaled)

    raw_score = -model.decision_function(X_scaled)

    result["dealer_anomaly_raw_score"] = raw_score

    mm = MinMaxScaler(feature_range=(0, 100))

    result["dealer_anomaly_score"] = mm.fit_transform(
        raw_score.reshape(-1, 1)
    )

    result["dealer_risk_segment"] = pd.cut(
        result["dealer_anomaly_score"],
        bins=[-1, 60, 80, 100],
        labels=["Low", "Medium", "High"]
    )

    return result


# ============================================================
# 14. MOTORCYCLE 2:
# DEALER RISK DRIVER EXPLANATIONS
# ============================================================

def add_dealer_risk_drivers(
    dealer_scored: pd.DataFrame
) -> pd.DataFrame:
    """
    Adds human-readable dealer risk reasons.

    Why:
    Business users need to understand why a dealer was flagged.
    This supports audit review, dashboard explainability, and stakeholder trust.
    """

    df = dealer_scored.copy()

    driver_rules = {
        "High dealer claim volume": "dealer_claim_count",
        "High claims per active day": "claims_per_active_day",
        "High rejection rate": "rejection_rate",
        "High after-hours rate": "after_hours_rate",
        "High claimant concentration": "claimant_hhi",
        "High claimant inequality": "claimant_gini",
        "High top-claimant dependency": "top_claimant_share",
        "Many high-risk claimants": "high_risk_claimant_count",
        "High average claimant risk": "avg_claimant_risk_score",
        "Temporal clustering": "max_claims_same_hour",
        "High 7-day dealer spike": "max_dealer_7d_claim_count",
        "High claims-to-sell-in units ratio": "max_claims_to_sell_in_units_ratio"
    }

    percentile_cols = []

    for label, col in driver_rules.items():
        if col in df.columns:
            pct_col = f"{col}_percentile"

            df[pct_col] = df[col].rank(pct=True)

            percentile_cols.append((label, pct_col))

    def top_drivers(row):
        scored = []

        for label, pct_col in percentile_cols:
            value = row[pct_col]

            if pd.notna(value):
                scored.append((label, value))

        scored = sorted(scored, key=lambda x: x[1], reverse=True)

        return "; ".join([x[0] for x in scored[:3]])

    df["top_dealer_risk_drivers"] = df.apply(top_drivers, axis=1)

    return df


# ============================================================
# 15. FINAL REVIEW QUEUE
# ============================================================

def build_review_queue(
    claims_df: pd.DataFrame,
    claimant_scored: pd.DataFrame,
    dealer_scored: pd.DataFrame
) -> pd.DataFrame:
    """
    Creates claim-level review queue.

    Why:
    Claimant and dealer scores are useful, but audit teams usually need
    a prioritized list of individual claims to review.

    Final score combines:
    - claimant risk
    - dealer risk
    - claim-level rule score
    """

    review = claims_df.copy()

    # Attach claimant risk to each claim.
    review = review.merge(
        claimant_scored[
            [
                "claimant_id",
                "claimant_anomaly_score",
                "claimant_risk_segment",
                "top_claimant_risk_drivers"
            ]
        ],
        on="claimant_id",
        how="left"
    )

    # Attach dealer risk to each claim.
    review = review.merge(
        dealer_scored[
            [
                "dealer_id",
                "dealer_anomaly_score",
                "dealer_risk_segment",
                "top_dealer_risk_drivers"
            ]
        ],
        on="dealer_id",
        how="left"
    )

    # Claim-level rule signals.
    # Percentile rank identifies unusually high claim amounts or quantities.
    review["claim_amount_percentile"] = (
        review["claim_amount"]
        .rank(pct=True)
    )

    review["quantity_percentile"] = (
        review["quantity"]
        .rank(pct=True)
    )

    # Lightweight claim-level rule score.
    # This can be expanded later with Bike-level validation rules.
    review["claim_level_rule_score"] = (
        20 * review["is_after_hours"].fillna(0) +
        20 * review["claim_amount_percentile"].fillna(0) +
        20 * review["quantity_percentile"].fillna(0)
    )

    # Final weighted fraud risk score.
    #
    # Weighting rationale:
    # - Claimant behavior gets the highest weight because Motorcycle 1 is focused
    #   on individual claimant behavior.
    # - Dealer behavior receives strong weight because Motorcycle 2 adds systemic
    #   context.
    # - Claim-level rules provide additional immediate signals.
    #
    # These weights are configurable and can later be exposed as dashboard sliders.
    review["final_fraud_risk_score"] = (
        0.45 * review["claimant_anomaly_score"].fillna(0) +
        0.35 * review["dealer_anomaly_score"].fillna(0) +
        0.20 * review["claim_level_rule_score"].fillna(0)
    )

    review["final_risk_segment"] = pd.cut(
        review["final_fraud_risk_score"],
        bins=[-1, 60, 80, 100],
        labels=["Low", "Medium", "High"]
    )

    review = review.sort_values(
        "final_fraud_risk_score",
        ascending=False
    )

    return review


# ============================================================
# 16. END-TO-END PIPELINE FUNCTION
# ============================================================

def run_claims_fraud_pipeline(
    claims: pd.DataFrame,
    audit_threshold: float = None,
    sell_in: pd.DataFrame = None,
    claimant_contamination: float = 0.05,
    dealer_contamination: float = 0.05
):
    """
    Runs the full fraud detection pipeline.

    Inputs:
    claims:
        Raw claims history dataframe.

    audit_threshold:
        Optional dollar threshold for detecting claims just below audit limit.

    sell_in:
        Optional sell-in / wholesale dataframe.
        If not available, dealer sell-in ratio features are skipped.

    claimant_contamination:
        Expected fraction of anomalous claimants.

    dealer_contamination:
        Expected fraction of anomalous dealers.

    Outputs:
    claims_clean:
        Cleaned claim-level data.

    claimant_scored:
        Claimant-level risk scoring output.

    dealer_scored:
        Dealer-level risk scoring output.

    review_queue:
        Claim-level prioritized audit queue.
    """

    # --------------------------------------------------------
    # Step 1: Clean and enrich raw claims data
    # --------------------------------------------------------
    claims_clean = prepare_claims_data(claims)

    # --------------------------------------------------------
    # Step 2: Create claimant-level features
    # --------------------------------------------------------
    claimant_features = build_claimant_features(
        claims_clean,
        audit_threshold=audit_threshold
    )

    # --------------------------------------------------------
    # Step 3: Add shared identity risk signals
    # --------------------------------------------------------
    claimant_features = add_shared_identity_features(
        claims_clean,
        claimant_features
    )

    # --------------------------------------------------------
    # Step 4: Add fuzzy duplicate signals for serial number
    # --------------------------------------------------------
    if "serial_number" in claims_clean.columns:
        serial_fuzzy = fuzzy_duplicate_score(
            claims_clean,
            text_col="serial_number",
            min_similarity=0.92
        )
    else:
        serial_fuzzy = claims_clean[["claim_id"]].copy()

    # --------------------------------------------------------
    # Step 5: Add fuzzy duplicate signals for claimant name
    # --------------------------------------------------------
    if "claimant_name" in claims_clean.columns:
        name_fuzzy = fuzzy_duplicate_score(
            claims_clean,
            text_col="claimant_name",
            min_similarity=0.90
        )
    else:
        name_fuzzy = claims_clean[["claim_id"]].copy()

    # --------------------------------------------------------
    # Step 6: Merge fuzzy features back to claim level
    # --------------------------------------------------------
    claims_with_fuzzy = (
        claims_clean
        .merge(serial_fuzzy, on="claim_id", how="left")
        .merge(name_fuzzy, on="claim_id", how="left")
    )

    # --------------------------------------------------------
    # Step 7: Roll fuzzy duplicate signals to claimant level
    # --------------------------------------------------------
    fuzzy_agg_dict = {}

    if "serial_number_max_similarity" in claims_with_fuzzy.columns:
        fuzzy_agg_dict["max_serial_similarity"] = (
            "serial_number_max_similarity",
            "max"
        )

    if "serial_number_near_duplicate_flag" in claims_with_fuzzy.columns:
        fuzzy_agg_dict["serial_near_duplicate_rate"] = (
            "serial_number_near_duplicate_flag",
            "mean"
        )

    if "claimant_name_max_similarity" in claims_with_fuzzy.columns:
        fuzzy_agg_dict["max_name_similarity"] = (
            "claimant_name_max_similarity",
            "max"
        )

    if "claimant_name_near_duplicate_flag" in claims_with_fuzzy.columns:
        fuzzy_agg_dict["name_near_duplicate_rate"] = (
            "claimant_name_near_duplicate_flag",
            "mean"
        )

    if len(fuzzy_agg_dict) > 0:
        fuzzy_claimant_features = (
            claims_with_fuzzy.groupby("claimant_id")
            .agg(**fuzzy_agg_dict)
            .reset_index()
        )

        claimant_features = claimant_features.merge(
            fuzzy_claimant_features,
            on="claimant_id",
            how="left"
        )

    claimant_features = claimant_features.fillna(0)

    # --------------------------------------------------------
    # Step 8: Score claimant anomalies
    # --------------------------------------------------------
    claimant_scored = score_claimant_anomalies(
        claimant_features,
        contamination=claimant_contamination
    )

    # --------------------------------------------------------
    # Step 9: Add claimant risk explanations
    # --------------------------------------------------------
    claimant_scored = add_claimant_risk_drivers(claimant_scored)

    # --------------------------------------------------------
    # Step 10: Build dealer features using claimant risk context
    # --------------------------------------------------------
    dealer_features = build_dealer_features(
        claims_clean,
        claimant_scored=claimant_scored
    )

    # --------------------------------------------------------
    # Step 11: Add dealer peer benchmarks
    # --------------------------------------------------------
    dealer_features = add_dealer_peer_benchmarks(dealer_features)

    # --------------------------------------------------------
    # Step 12: Optional sell-in / wholesale comparison
    # --------------------------------------------------------
    if sell_in is not None:
        dealer_features = add_sell_in_claim_ratio(
            claims_clean,
            dealer_features,
            sell_in
        )

    # --------------------------------------------------------
    # Step 13: Score dealer anomalies
    # --------------------------------------------------------
    dealer_scored = score_dealer_anomalies(
        dealer_features,
        contamination=dealer_contamination
    )

    # --------------------------------------------------------
    # Step 14: Add dealer risk explanations
    # --------------------------------------------------------
    dealer_scored = add_dealer_risk_drivers(dealer_scored)

    # --------------------------------------------------------
    # Step 15: Create final claim review queue
    # --------------------------------------------------------
    review_queue = build_review_queue(
        claims_clean,
        claimant_scored,
        dealer_scored
    )

    return claims_clean, claimant_scored, dealer_scored, review_queue


# ============================================================
# 17. HOW TO RUN THE PIPELINE
# ============================================================

# Example:
#
# claims_clean, claimant_scored, dealer_scored, review_queue = run_claims_fraud_pipeline(
#     claims=claims,
#     audit_threshold=500,
#     sell_in=None,
#     claimant_contamination=0.05,
#     dealer_contamination=0.05
# )


# ============================================================
# 18. SAVE OUTPUTS FOR DASHBOARDING OR POWER BI
# ============================================================

# After running the pipeline, you can save the outputs:
#
# claimant_scored.to_csv("claimant_risk_scores.csv", index=False)
# dealer_scored.to_csv("dealer_risk_scores.csv", index=False)
# review_queue.to_csv("fraud_review_queue.csv", index=False)


# ============================================================
# 19. RECOMMENDED DASHBOARD VIEWS
# ============================================================

# View 1: Claimant risk table
#
# claimant_scored[
#     [
#         "claimant_id",
#         "claim_count",
#         "claimant_anomaly_score",
#         "claimant_risk_segment",
#         "top_claimant_risk_drivers"
#     ]
# ].sort_values("claimant_anomaly_score", ascending=False).head(25)


# View 2: Dealer risk table
#
# dealer_scored[
#     [
#         "dealer_id",
#         "dealer_region",
#         "dealer_claim_count",
#         "dealer_anomaly_score",
#         "dealer_risk_segment",
#         "top_dealer_risk_drivers"
#     ]
# ].sort_values("dealer_anomaly_score", ascending=False).head(25)


# View 3: Claim review queue
#
# review_queue[
#     [
#         "claim_id",
#         "claimant_id",
#         "dealer_id",
#         "claim_amount",
#         "quantity",
#         "claimant_anomaly_score",
#         "dealer_anomaly_score",
#         "claim_level_rule_score",
#         "final_fraud_risk_score",
#         "final_risk_segment",
#         "top_claimant_risk_drivers",
#         "top_dealer_risk_drivers"
#     ]
# ].head(50)